# Deep SRQ Joint-Q Network Architecture Tuning

Tunes the hidden architecture of the DeepSRQ dueling joint-Q critic while holding the SRE and rollout setup fixed.

Fixed settings:
- Q-network family: `joint_output`
- Rollout mode: serial
- SRE backend: PATH C pool, `path_c_pool`
- PATH workers: 8
- PATH starts: 5 random restarts plus pure starts
- Robust epsilon: starts at 0.5 with a linear schedule

Set `N_EPISODES` lower for a smoke run before launching the full tuning grid.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "bimatrix_game":
    BIMATRIX_DIR = ROOT
else:
    BIMATRIX_DIR = ROOT / "discrete_action_space" / "bimatrix_game"
DISCRETE_DIR = BIMATRIX_DIR.parent
for path in (BIMATRIX_DIR, DISCRETE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

MPLCONFIGDIR = Path("/tmp") / "sre_dqn_matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

from experiment_harness import (
    BASE_SEED,
    DuelingDoubleDqnSreAgentConfig,
    run_deep_srq_ablation_variants,
    summarize_ablation_timing_rows,
)
from stats_utils import save_training_stats


In [ ]:
OUTPUT_ROOT = BIMATRIX_DIR / "ablation_runs" / "joint_q_architecture_tuning"

SCENARIOS = ("scenario1", "scenario3")
N_EPISODES = 3000
USE_GPU = True
WRITE_PLOTS = False

ROBUST_EPSILON_START = 0.5
EPSILON_SCHEDULE = "linear"

FIXED_HYPERPARAMS = DuelingDoubleDqnSreAgentConfig(
    network_type="joint_output",
    q_hidden_dims=(128, 128),
    sre_solver_name="path_c_pool",
    sre_solver_workers=8,
    sre_num_repeats=5,
    sre_include_pure_starts=True,
    epsilon_schedule=EPSILON_SCHEDULE,
    epsilon_robust_initial=ROBUST_EPSILON_START,
)

RUN_VARIANTS = [
    {
        "label": "joint_64x64",
        "display_name": "Joint Q 64x64",
        "hyperparameter_overrides": {"q_hidden_dims": (64, 64)},
    },
    {
        "label": "joint_128x64",
        "display_name": "Joint Q 128x64",
        "hyperparameter_overrides": {"q_hidden_dims": (128, 64)},
    },
    {
        "label": "joint_128x128",
        "display_name": "Joint Q 128x128",
        "hyperparameter_overrides": {"q_hidden_dims": (128, 128)},
    },
    {
        "label": "joint_256x128",
        "display_name": "Joint Q 256x128",
        "hyperparameter_overrides": {"q_hidden_dims": (256, 128)},
    },
    {
        "label": "joint_256x256",
        "display_name": "Joint Q 256x256",
        "hyperparameter_overrides": {"q_hidden_dims": (256, 256)},
    },
    {
        "label": "joint_128x128x128",
        "display_name": "Joint Q 128x128x128",
        "hyperparameter_overrides": {"q_hidden_dims": (128, 128, 128)},
    },
]

VARIANT_BY_LABEL = {variant["label"]: variant for variant in RUN_VARIANTS}
VARIANT_ORDER = {variant["label"]: idx for idx, variant in enumerate(RUN_VARIANTS)}
DISPLAY_NAMES = {variant["label"]: variant["display_name"] for variant in RUN_VARIANTS}


In [ ]:
results = run_deep_srq_ablation_variants(
    variants=RUN_VARIANTS,
    scenarios=SCENARIOS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    use_gpu=USE_GPU,
    write_plots=WRITE_PLOTS,
    n_episodes=N_EPISODES,
    hyperparameters=FIXED_HYPERPARAMS,
)

save_training_stats(OUTPUT_ROOT / "joint_q_architecture_tuning_raw_manifest.txt", results)


In [ ]:
def _fmt(value, kind):
    if value is None:
        return "-"
    if kind == "int":
        return f"{int(value):,}"
    if kind == "float1":
        return f"{float(value):,.1f}"
    if kind == "float2":
        return f"{float(value):,.2f}"
    if kind == "dims":
        return "x".join(str(int(dim)) for dim in value)
    return str(value)


def print_pretty_table(rows, columns):
    rendered = []
    headers = [title for _, title, _ in columns]
    for row in rows:
        rendered.append([_fmt(row.get(key), kind) for key, _, kind in columns])
    widths = [len(header) for header in headers]
    for row in rendered:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    header_line = " | ".join(header.ljust(width) for header, width in zip(headers, widths))
    separator = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(separator)
    for row in rendered:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))


summary_rows = []
for row in summarize_ablation_timing_rows(results):
    stats = results[row["scenario"]][row["variant"]]
    hp = stats["hyperparameters"]
    epsilon_cfg = stats["agent_epsilon_configs"][0]
    summary_rows.append({
        **row,
        "architecture": DISPLAY_NAMES[row["variant"]],
        "q_hidden_dims": hp.get("q_hidden_dims"),
        "network_type": hp.get("network_type"),
        "epsilon": epsilon_cfg["epsilon_robust_initial"],
        "schedule": epsilon_cfg["epsilon_schedule"],
        "restarts": hp.get("sre_num_repeats"),
        "pure_starts": hp.get("sre_include_pure_starts"),
        "workers": hp.get("sre_solver_workers"),
        "agent_joint_mean_last": row["agent1_mean_last"] + row["agent2_mean_last"],
    })

summary_rows.sort(key=lambda row: (row["scenario"], VARIANT_ORDER[row["variant"]]))

print_pretty_table(
    summary_rows,
    columns=(
        ("scenario", "Scenario", "str"),
        ("architecture", "Architecture", "str"),
        ("q_hidden_dims", "Hidden", "dims"),
        ("network_type", "Network", "str"),
        ("epsilon", "Eps0", "float2"),
        ("schedule", "Schedule", "str"),
        ("pure_starts", "Pure", "str"),
        ("restarts", "Random", "int"),
        ("workers", "Workers", "int"),
        ("env_steps", "Env Steps", "int"),
        ("wall_seconds", "Wall s", "float1"),
        ("steps_per_second", "Steps/s", "float2"),
        ("mean_sre_ms", "SRE ms", "float2"),
        ("mean_backend_ms", "Backend ms", "float2"),
        ("agent1_mean_last", "A1 Last", "float2"),
        ("agent2_mean_last", "A2 Last", "float2"),
        ("agent_joint_mean_last", "Joint Last", "float2"),
    ),
)

save_training_stats(
    OUTPUT_ROOT / "joint_q_architecture_tuning_summary.txt",
    {"summary_rows": summary_rows},
)


In [ ]:
best_rows = []
for scenario_key in SCENARIOS:
    scenario_rows = [row for row in summary_rows if row["scenario"] == scenario_key]
    best_rows.append(max(scenario_rows, key=lambda row: row["agent_joint_mean_last"]))

print_pretty_table(
    best_rows,
    columns=(
        ("scenario", "Scenario", "str"),
        ("architecture", "Best Architecture", "str"),
        ("q_hidden_dims", "Hidden", "dims"),
        ("agent_joint_mean_last", "Joint Last", "float2"),
        ("wall_seconds", "Wall s", "float1"),
        ("steps_per_second", "Steps/s", "float2"),
    ),
)

save_training_stats(
    OUTPUT_ROOT / "joint_q_architecture_tuning_best_by_scenario.txt",
    {"best_rows": best_rows},
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def rolling_mean(values, window=100):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(values, kernel, mode="valid")


ROLLING_WINDOW = 100
COLORS = dict(zip(
    [variant["label"] for variant in RUN_VARIANTS],
    plt.rcParams["axes.prop_cycle"].by_key()["color"],
))

for scenario_key in SCENARIOS:
    fig, ax = plt.subplots(figsize=(10, 5))
    for variant in RUN_VARIANTS:
        stats = results[scenario_key][variant["label"]]
        rewards = stats["rewards"]
        min_len = min(len(rewards[0]), len(rewards[1]))
        if min_len == 0:
            continue
        joint_reward = (
            np.asarray(rewards[0][:min_len], dtype=float)
            + np.asarray(rewards[1][:min_len], dtype=float)
        )
        smoothed = rolling_mean(joint_reward, window=ROLLING_WINDOW)
        x = np.arange(smoothed.size) + min(ROLLING_WINDOW, joint_reward.size)
        ax.plot(
            x,
            smoothed,
            label=variant["display_name"],
            color=COLORS.get(variant["label"]),
        )
    ax.set_title(
        f"{scenario_key} | joint_output | eps0={ROBUST_EPSILON_START:g} | "
        f"{EPSILON_SCHEDULE} | PATH pool x{FIXED_HYPERPARAMS.sre_solver_workers} | "
        f"pure + {FIXED_HYPERPARAMS.sre_num_repeats} random"
    )
    ax.set_xlabel("Completed episodes")
    ax.set_ylabel(f"Joint reward, rolling mean ({ROLLING_WINDOW})")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f"reward_curves_{scenario_key}.png")
    plt.show()


In [ ]:
configured_runs = len(SCENARIOS) * len(RUN_VARIANTS)
print(f"Configured total training runs: {configured_runs}")
print(f"Scenarios: {', '.join(SCENARIOS)}")
print("Fixed critic family: joint_output")
print(f"PATH workers: {FIXED_HYPERPARAMS.sre_solver_workers}")
print(
    "PATH starts: "
    f"pure_starts={FIXED_HYPERPARAMS.sre_include_pure_starts}, "
    f"random_restarts={FIXED_HYPERPARAMS.sre_num_repeats}"
)
print(f"Architecture variants: {', '.join(variant['label'] for variant in RUN_VARIANTS)}")
